In [1305]:
import torch
import torch.nn.functional as F
import torch.nn as nn

In [1306]:
torch.manual_seed(3);

In [1307]:
a = torch.tril(torch.ones(3, 3))
a

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [1308]:
a = a / a.sum(dim=1, keepdim=True)
a

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

In [1309]:
b = torch.tensor([[2.0], [6.0], [4.0],])
print(b.shape)
b

torch.Size([3, 1])


tensor([[2.],
        [6.],
        [4.]])

In [1310]:
c = a @ b
print(c.shape)
c

torch.Size([3, 1])


tensor([[2.],
        [4.],
        [4.]])

In [1311]:
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(dim=1, keepdim=True)
r = wei @ x
r.shape

torch.Size([4, 8, 2])

(T,T) @ (B,T,C) broadcast to (B,T,T) @ (B,T,C), so it ran 4 separate matmuls, one per sequence in the batch.

In [1312]:
xbow = torch.zeros(B, T, C)
for b in range(B):
    for t in range(T):
        xbow[b, t] = x[b, :t+1].mean(dim=0)
torch.allclose(xbow, r, atol=1e-6)

True

In [1313]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [1314]:
wei = F.softmax(wei, dim=-1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [1315]:
for i in range(T):
    print(wei[i].sum())

tensor(1.)
tensor(1.)
tensor(1.)
tensor(1.)
tensor(1.)
tensor(1.)
tensor(1.)
tensor(1.)


In [1316]:
tril = torch.tril(torch.ones(T, T))
wei = torch.randn(T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei

tensor([[-0.0752,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.5125, -0.0198,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.1870, -0.1813, -0.0914,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.3290,  0.3323,  0.3025, -0.9812,    -inf,    -inf,    -inf,    -inf],
        [-0.4964, -0.6016, -0.4775, -0.2988,  0.3932,    -inf,    -inf,    -inf],
        [-0.3282,  1.6216,  0.8594,  0.6996, -1.2667,  1.2725,    -inf,    -inf],
        [-0.3815,  0.5052,  0.5831, -0.0215,  2.2463, -0.3071, -0.7002,    -inf],
        [-0.4709,  0.0302,  0.1596,  0.3998,  0.0178,  2.3573, -0.7966,  0.1854]])

In [1317]:
wei = F.softmax(wei, dim=-1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6300, 0.3700, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4084, 0.2825, 0.3091, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3080, 0.3090, 0.2999, 0.0831, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1522, 0.1370, 0.1551, 0.1854, 0.3704, 0.0000, 0.0000, 0.0000],
        [0.0514, 0.3613, 0.1686, 0.1437, 0.0201, 0.2549, 0.0000, 0.0000],
        [0.0432, 0.1049, 0.1134, 0.0620, 0.5984, 0.0466, 0.0314, 0.0000],
        [0.0356, 0.0587, 0.0668, 0.0850, 0.0580, 0.6017, 0.0257, 0.0686]])

In [1318]:
for i in range(T):
    print(wei[i].sum())

tensor(1.)
tensor(1.)
tensor(1.0000)
tensor(1.)
tensor(1.)
tensor(1.)
tensor(1.0000)
tensor(1.)


each still sums to 1 and still can't see the future. That's the shape of attention.

Only problem: those numbers are random. They should come from the tokens.

In [1319]:
head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)

k = key(x)      # (B, T, 16)
q = query(x)    # (B, T, 16)
wei = q @ k.transpose(-2, -1)
wei.shape

torch.Size([4, 8, 8])

(B, T, T) — and now it's per-batch. Each of the 4 sequences got its own score matrix, computed from its own tokens. The random version had one matrix shared by all.

wei[b, i, j] is the dot product of query i with key j: how much token i cares about token j.

In [1320]:
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
print(wei[0])

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4368, 0.5632, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3370, 0.2618, 0.4013, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0425, 0.0199, 0.1403, 0.7973, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1532, 0.1482, 0.2002, 0.3174, 0.1810, 0.0000, 0.0000, 0.0000],
        [0.1495, 0.1336, 0.1762, 0.2227, 0.1551, 0.1630, 0.0000, 0.0000],
        [0.1843, 0.1435, 0.1404, 0.0756, 0.1270, 0.1382, 0.1910, 0.0000],
        [0.1646, 0.1473, 0.1084, 0.0484, 0.1130, 0.1159, 0.1403, 0.1621]],
       grad_fn=<SelectBackward0>)


In [1321]:
value = nn.Linear(C, head_size, bias=False)
v = value(x)
out = wei @ v
out.shape

torch.Size([4, 8, 16])

v instead of raw x because a token's identity and what it offers to others are different things — the key says "I'm a noun", the value says "here's the content you get if you attend to me."

In [1322]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)
wei = q @ k.transpose(-2, -1)
print(k.var(), q.var(), wei.var())

tensor(1.0227) tensor(1.0197) tensor(16.0828)


≈1, ≈1, ≈16. The dot product sums head_size products, so the variance scales with head_size (16).

In [1323]:
t = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print(f'{t.var():.2f}')
print(F.softmax(t, dim=-1))
print(f'{(t * 8).var():.2f}')
print(F.softmax(t * 8, dim=-1))

0.09
tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
6.08
tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


Same relative ordering, but scaled up it collapses onto one entry (0.80). A peaked softmax means each token attends to exactly one other token — and the near-zero entries barely pass gradient.

At initialization you want diffuse, so information flows from everywhere. 

In [1324]:
print(wei[0][3])
print(F.softmax(wei[0][3], dim=-1))

tensor([-0.3693,  4.9880, -1.5419,  0.9055,  2.0617,  1.1126, -1.0549,  0.0327])
tensor([0.0043, 0.9035, 0.0013, 0.0152, 0.0484, 0.0187, 0.0021, 0.0064])


probabilites are very high for for larger values, while negligble for others

In [1325]:
wei = q @ k.transpose(-2, -1) * head_size**-0.5
wei.var()

tensor(1.0052)

Expect ≈1. That's the "scaled" in scaled dot-product attention

In [1326]:
print(wei[0][3])
print(F.softmax(wei[0][3], dim=-1))

tensor([-0.0923,  1.2470, -0.3855,  0.2264,  0.5154,  0.2782, -0.2637,  0.0082])
tensor([0.0822, 0.3136, 0.0613, 0.1130, 0.1509, 0.1190, 0.0692, 0.0909])


probabilites are more uniform